# CEO Reputational Risk & NVIDIA — Quantitative Research Lab

**Event-driven research into abnormal returns, volatility, trading activity, peer performance and portfolio stress scenarios.**

This notebook is designed as a reproducible research artifact rather than a collection of introductory charts. It uses interactive Plotly visualizations, explicit data-quality checks, multiple expected-return benchmarks, robust inference, placebo diagnostics and transparent scenario assumptions.

> **Identification rule:** observed market-cap changes, model-estimated abnormal returns and causal economic damage are three different quantities. This case study estimates the first two; it does not automatically identify the third.

| Module | Methods | Deliverables |
|---|---|---|
| Market reaction | Indexed wealth, drawdowns, return distributions | Interactive price & risk dashboard |
| Event study | Market model, peer-control model, AR/CAR | Event-time plots and event-window table |
| Inference | HAC standard errors, residual block bootstrap, placebo dates, FDR | Robustness diagnostics |
| Risk transmission | Realized/downside volatility, rolling beta, volume surprise, GARCH | Risk-regime dashboard |
| Cross-section | Correlation, rolling dependence, peer risk-return map | Comparative analytics |
| Portfolio lab | Deterministic stress, correlated Monte Carlo, sensitivity | Loss distribution and scenario analysis |
| Optional text | FinBERT headline sentiment with publication-time mapping | News/market alignment |

**Source-verification gate:** event dates in the supplied notebook are provisional and lack source URLs. They are therefore disabled below. Add original source URLs, verify the first public timestamp and only then enable events. No unverified date should enter an event study.

## 0. Environment, reproducibility and research controls

In [ ]:
# If needed: %pip install -q yfinance pandas numpy scipy statsmodels arch plotly scikit-learn

import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR, FIG_DIR = Path("data"), Path("visualizations")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

TEMPLATE = "plotly_dark"
C = {"nvda":"#76B900", "market":"#4EA5D9", "positive":"#2EC4B6",
     "negative":"#FF5A5F", "accent":"#B79CED", "muted":"#9AA4B2"}

In [ ]:
STOCKS = ["NVDA", "META", "AMZN", "GOOGL", "TSLA", "AMD", "AVGO", "MSFT"]
BENCHMARKS = ["SPY", "QQQ", "SOXX"]
TICKERS = STOCKS + BENCHMARKS
TARGET, MARKET = "NVDA", "SPY"

START_DATE = "2025-11-01"
END_DATE = "2026-11-15"   # yfinance end is exclusive

ESTIMATION_LENGTH = 120
ESTIMATION_GAP = 10
EVENT_WINDOWS = [(-1, 1), (-3, 3), (-5, 5), (0, 0)]
BOOTSTRAP_REPS = 3000
PLACEBO_REPS = 500
BLOCK_LENGTH = 3

# Inherited provisional dates: verify timestamps and source URLs before enabling.
events = pd.DataFrame([
    dict(event_id="E1", event="CEO response / board investigation (provisional)",
         announcement_date="2026-11-08", day0="2026-11-09",
         source_name="", source_url="", verified=False, include=False,
         notes="Verify original public timestamp and whether news arrived before/after market close."),
    dict(event_id="E2", event="Further reporting / leadership announcement (provisional)",
         announcement_date="2026-11-12", day0="2026-11-12",
         source_name="", source_url="", verified=False, include=False,
         notes="Verify each announcement. Overlapping information cannot be separated with daily data.")
])
events["announcement_date"] = pd.to_datetime(events["announcement_date"])
events["day0"] = pd.to_datetime(events["day0"])

### Design decisions

- Day 0 is the first trading session able to reflect the public announcement. Weekend and after-close news generally map to the next trading session.
- The estimation window contains 120 trading sessions and ends 10 sessions before day 0. This is a design choice; test alternatives.
- The preferred control sensitivity is an equal-weight basket of non-NVIDIA peers. SPY is the broad-market factor.
- SOXX contains NVIDIA. A model including SOXX may absorb part of NVIDIA's own event move, so it is a *sensitivity specification*, not an independent counterfactual.
- Closely timed announcements are interpreted as a combined information event when daily data cannot separate them.
- Report incomplete windows, adjust for multiple testing and avoid causal language.

## 1. Data acquisition and quality control

In [ ]:
def download_market_data(tickers, start, end):
    files = {k: DATA_DIR/f"{k}.csv" for k in ["close","volume","open","high","low"]}
    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True,
                          actions=False, progress=False, group_by="column", threads=True)
        if raw.empty:
            raise RuntimeError("Yahoo Finance returned an empty dataset.")
        if not isinstance(raw.columns, pd.MultiIndex):
            raise ValueError("Expected multi-ticker MultiIndex response.")
        frames = {k: raw[k].copy() for k in ["Close","Volume","Open","High","Low"]}
        frames = {k: v for k,v in zip(["close","volume","open","high","low"], frames.values())}
        for k, frame in frames.items():
            frame.index = pd.to_datetime(frame.index).tz_localize(None)
            frame.sort_index(inplace=True)
            frame.to_csv(files[k])
        return frames, "Yahoo Finance download"
    except Exception as exc:
        if not all(p.exists() for p in files.values()):
            raise RuntimeError(f"Download failed ({exc}); no complete local cache found in ./data.")
        frames = {k: pd.read_csv(p, index_col=0, parse_dates=True) for k,p in files.items()}
        return frames, "Local CSV cache"

data, DATA_SOURCE = download_market_data(TICKERS, START_DATE, END_DATE)
prices, volume = data["close"], data["volume"]
open_px, high_px, low_px = data["open"], data["high"], data["low"]

available = [t for t in TICKERS if t in prices.columns]
prices, volume = prices[available], volume.reindex(columns=available)
returns = prices.pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan)
log_returns = np.log(prices).diff()
returns = returns.dropna(subset=[TARGET, MARKET])

quality = pd.DataFrame({
    "observations": prices.notna().sum(),
    "missing_pct": prices.isna().mean()*100,
    "latest_date": [prices[c].last_valid_index() for c in prices],
    "nonpositive_prices": [(prices[c].dropna()<=0).sum() for c in prices]
})
display(quality)
print(f"Source: {DATA_SOURCE} | {prices.index.min().date()} to {prices.index.max().date()} | {len(prices):,} sessions")

In [ ]:
assert TARGET in prices and MARKET in prices, "Target or market benchmark missing."
assert prices[TARGET].notna().sum() > 150, "Insufficient history for event-study estimation."
print("Missing tickers:", sorted(set(TICKERS)-set(prices.columns)))
print("Duplicate dates:", int(prices.index.duplicated().sum()))
print("Returns sample:")
display(returns.tail())

## 2. Event ledger and verification gate

In [ ]:
events["source_present"] = events["source_url"].fillna("").str.startswith(("https://","http://"))
events["analysis_ready"] = events["verified"].astype(bool) & events["include"].astype(bool) & events["source_present"]
display(events[["event_id","event","announcement_date","day0","source_name","source_url",
                "verified","include","analysis_ready","notes"]])

active_events = events.loc[events["analysis_ready"]].copy().reset_index(drop=True)

def map_to_session(calendar, date):
    pos = calendar.searchsorted(pd.Timestamp(date))
    return calendar[pos] if pos < len(calendar) else pd.NaT

if active_events.empty:
    print("EVENT STUDY DISABLED: verify each source URL/timestamp and set verified=True, include=True.")
else:
    active_events["mapped_day0"] = [map_to_session(returns.index,d) for d in active_events["day0"]]
    display(active_events[["event_id","event","day0","mapped_day0"]])

## 3. Market reaction: indexed wealth, drawdowns and distribution shape

In [ ]:
focus = [t for t in ["NVDA","AMD","AVGO","SOXX","QQQ","SPY"] if t in prices]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.08,
                    row_heights=[.62,.38],
                    subplot_titles=("Adjusted closing price","Indexed wealth: first observation = 100"))
for t in focus:
    color = C["nvda"] if t==TARGET else None
    fig.add_trace(go.Scatter(x=prices.index,y=prices[t],name=f"{t} price",
                             line=dict(width=2.8 if t==TARGET else 1.5,color=color)),row=1,col=1)
    idx = prices[t]/prices[t].dropna().iloc[0]*100
    fig.add_trace(go.Scatter(x=idx.index,y=idx,name=f"{t} indexed",
                             line=dict(width=2.8 if t==TARGET else 1.5,color=color),
                             showlegend=False),row=2,col=1)
for _, ev in active_events.iterrows():
    for row in [1,2]:
        fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"],row=row,col=1)
fig.update_layout(template=TEMPLATE,height=820,title="NVIDIA vs semiconductor and broad-market benchmarks",
                  hovermode="x unified",legend=dict(orientation="h",y=-.13))
fig.update_yaxes(title="USD (adjusted)",row=1,col=1)
fig.update_yaxes(title="Indexed price",row=2,col=1)
fig.show()

In [ ]:
wealth = (1+returns.fillna(0)).cumprod()
drawdowns = wealth/wealth.cummax()-1

fig = go.Figure()
for t in [x for x in STOCKS if x in drawdowns]:
    fig.add_trace(go.Scatter(x=drawdowns.index,y=drawdowns[t]*100,name=t,
                             line=dict(width=2.5 if t==TARGET else 1.2)))
for _,ev in active_events.iterrows():
    fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"])
fig.update_layout(template=TEMPLATE,title="Underwater chart: drawdown from prior peak",
                  yaxis_title="Drawdown (%)",height=520,hovermode="x unified")
fig.show()

dd_summary = pd.DataFrame({
    "max_drawdown":drawdowns.min(),
    "max_drawdown_date":drawdowns.idxmin(),
    "latest_drawdown":drawdowns.iloc[-1]
}).sort_values("max_drawdown")
display(dd_summary.style.format({"max_drawdown":"{:.1%}","latest_drawdown":"{:.1%}"}))

In [ ]:
fig = make_subplots(rows=1,cols=2,subplot_titles=("Daily return distributions","5th-percentile downside return"))
for t in [x for x in STOCKS if x in returns]:
    fig.add_trace(go.Histogram(x=returns[t].dropna()*100,name=t,opacity=.42,
                               histnorm="probability density"),row=1,col=1)
    fig.add_trace(go.Bar(x=[t],y=[returns[t].quantile(.05)*100],name=t,
                         showlegend=False),row=1,col=2)
fig.update_layout(template=TEMPLATE,barmode="overlay",height=480,
                  title="Return distribution and downside-tail diagnostics")
fig.update_xaxes(title="Daily return (%)",row=1,col=1)
fig.update_yaxes(title="Density",row=1,col=1)
fig.update_yaxes(title="5th percentile daily return (%)",row=1,col=2)
fig.show()

## 4. Event study engine

Expected-return specifications:

1. **Market model:** target return explained by SPY.
2. **Peer-control model:** target return explained by the equal-weight return of available peers excluding NVIDIA.
3. **Market + sector sensitivity:** SPY plus SOXX; interpret cautiously because SOXX holds NVIDIA.

For each event, calculate AR and CAR across several windows. Inference includes HAC covariance for estimated parameters, an approximate prediction-error interval, and a residual moving-block bootstrap under a zero-abnormal-return null. These are robustness tools, not causal identification.

In [ ]:
def peer_control_series(ret):
    peers=[t for t in STOCKS if t!=TARGET and t in ret.columns]
    if len(peers)<3: raise ValueError("Need at least three peer tickers.")
    return ret[peers].mean(axis=1).rename("PEER_EQW")

def factor_frame(ret, spec):
    if spec=="market": return ret[[MARKET]].copy()
    if spec=="market_sector": return ret[[MARKET,"SOXX"]].copy()
    if spec=="peer_control": return peer_control_series(ret).to_frame()
    raise ValueError(f"Unknown model: {spec}")

def fit_event_model(ret, ticker, day0, spec):
    pos=ret.index.searchsorted(pd.Timestamp(day0))
    est_end=pos-ESTIMATION_GAP
    est_start=est_end-ESTIMATION_LENGTH
    est=ret.iloc[max(0,est_start):max(0,est_end)]
    factors=factor_frame(ret,spec)
    fit=pd.concat([ret[ticker].rename("y"),factors],axis=1).dropna()
    fit=fit.loc[est.index.intersection(fit.index)]
    if len(fit)<max(60,20*(fit.shape[1]-1)):
        raise ValueError(f"Insufficient estimation sample: {len(fit)}")
    model=sm.OLS(fit["y"],sm.add_constant(fit.drop(columns="y"))).fit()
    return model,factors

def event_ar(ret,ticker,day0,spec,pre=10,post=10):
    model,factors=fit_event_model(ret,ticker,day0,spec)
    pos=ret.index.searchsorted(pd.Timestamp(day0))
    lo,hi=max(0,pos-pre),min(len(ret),pos+post+1)
    dates=ret.index[lo:hi]
    X=sm.add_constant(factors.loc[dates],has_constant="add").reindex(columns=model.params.index)
    ar=ret.loc[dates,ticker]-(X@model.params)
    rel=np.arange(lo-pos,hi-pos)
    ar.index=rel
    return ar,model,factors.loc[dates],dates

def block_bootstrap_p(residuals, observed_car, n=BOOTSTRAP_REPS, block=BLOCK_LENGTH):
    resid=np.asarray(residuals)
    if len(resid)<block+1: return np.nan
    L=len(observed_car) if hasattr(observed_car,"__len__") else int(observed_car[1])
    observed=float(observed_car.sum()) if hasattr(observed_car,"sum") else float(observed_car)
    sims=np.empty(n)
    for b in range(n):
        sample=[]
        while len(sample)<L:
            start=rng.integers(0,len(resid)-block+1)
            sample.extend(resid[start:start+block])
        sims[b]=np.sum(sample[:L])
    return (1+np.sum(np.abs(sims)>=abs(observed)))/(n+1)

def event_window_table(ret,ticker,day0,event_id,spec):
    max_pre=max(abs(a) for a,b in EVENT_WINDOWS)
    max_post=max(b for a,b in EVENT_WINDOWS)
    ar,model,factors,dates=event_ar(ret,ticker,day0,spec,max_pre,max_post)
    results=[]
    for a,b in EVENT_WINDOWS:
        mask=(ar.index>=a)&(ar.index<=b)
        selected=ar.loc[mask]
        L=len(selected)
        if L==0:
            continue
        # Map selected relative sessions to actual factor rows.
        selected_dates=dates[mask]
        Xw=sm.add_constant(factors.loc[selected_dates],has_constant="add").reindex(columns=model.params.index)
        c=Xw.sum(axis=0).to_numpy()
        robust=model.get_robustcov_results(cov_type="HAC",maxlags=3)
        cov=np.asarray(robust.cov_params())
        parameter_var=float(c@cov@c)
        residual_var=float(model.mse_resid)*L
        se=float(np.sqrt(max(parameter_var+residual_var,0)))
        car=float(selected.sum())
        tstat=car/se if se>0 else np.nan
        pval=2*st.t.sf(abs(tstat),df=max(int(model.df_resid),1)) if np.isfinite(tstat) else np.nan
        boot_p=block_bootstrap_p(model.resid,selected,n=BOOTSTRAP_REPS)
        results.append({
            "event_id":event_id,"ticker":ticker,"model":spec,"window":f"[{a},{b}]",
            "days_used":L,"expected_days":b-a+1,"complete_window":L==b-a+1,
            "CAR":car,"SE":se,"t_stat":tstat,"p_value":pval,
            "ci_low":car-1.96*se,"ci_high":car+1.96*se,
            "bootstrap_p":boot_p,"estimation_n":int(model.nobs),"estimation_R2":model.rsquared
        })
    return pd.DataFrame(results),ar,model

In [ ]:
event_results=[]; ar_results={}; fitted_models={}
if active_events.empty:
    print("No event study results: event ledger has no verified, enabled rows.")
else:
    for _,ev in active_events.iterrows():
        for spec in ["market","peer_control","market_sector"]:
            try:
                tab,ar,fit=event_window_table(returns,TARGET,ev["mapped_day0"],ev["event_id"],spec)
                event_results.append(tab)
                ar_results[(ev["event_id"],spec)]=ar
                fitted_models[(ev["event_id"],spec)]=fit
            except Exception as exc:
                print(f"Skipped {ev['event_id']} / {spec}: {exc}")
    event_results=pd.concat(event_results,ignore_index=True) if event_results else pd.DataFrame()

if not event_results.empty:
    valid=event_results["p_value"].notna()
    event_results["p_adj_BH"]=np.nan
    if valid.any():
        event_results.loc[valid,"p_adj_BH"]=multipletests(event_results.loc[valid,"p_value"],method="fdr_bh")[1]
    display(event_results.style.format({
        "CAR":"{:.2%}","SE":"{:.2%}","ci_low":"{:.2%}","ci_high":"{:.2%}",
        "p_value":"{:.4f}","p_adj_BH":"{:.4f}","bootstrap_p":"{:.4f}","estimation_R2":"{:.3f}"
    }))

In [ ]:
# Event-time panels: daily abnormal return and cumulative abnormal return.
if ar_results:
    for (event_id,spec),ar in ar_results.items():
        fit=fitted_models[(event_id,spec)]
        car=ar.cumsum()
        daily_sigma=np.sqrt(fit.mse_resid)
        t=np.arange(1,len(car)+1)
        band=1.96*daily_sigma*np.sqrt(t)
        fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.09,
                          subplot_titles=("Daily abnormal return","Cumulative abnormal return"))
        fig.add_trace(go.Bar(x=ar.index,y=ar.values*100,name="AR",
            marker_color=[C["negative"] if x<0 else C["positive"] for x in ar.values]),row=1,col=1)
        fig.add_hline(y=0,line_color="white",row=1,col=1)
        fig.add_trace(go.Scatter(x=car.index,y=car.values*100,name="CAR",
            line=dict(color=C["nvda"],width=3)),row=2,col=1)
        fig.add_trace(go.Scatter(x=car.index,y=(car.values+band)*100,
            line=dict(width=0),showlegend=False,hoverinfo="skip"),row=2,col=1)
        fig.add_trace(go.Scatter(x=car.index,y=(car.values-band)*100,
            line=dict(width=0),fill="tonexty",fillcolor="rgba(118,185,0,.12)",
            name="Approx. 95% band",hoverinfo="skip"),row=2,col=1)
        fig.add_vline(x=0,line_dash="dash",line_color=C["negative"])
        fig.update_layout(template=TEMPLATE,height=680,
            title=f"{event_id} | {spec} specification | event-time response",hovermode="x unified")
        fig.update_xaxes(title="Trading sessions from event day",row=2,col=1)
        fig.update_yaxes(title="AR (%)",row=1,col=1)
        fig.update_yaxes(title="CAR (%)",row=2,col=1)
        fig.show()
else:
    print("Enable verified events to render event-time plots.")

In [ ]:
# Placebo distribution: compare actual [-1,+1] CAR with pseudo-events before the event.
def placebo_distribution(ret,real_day0,spec="market",reps=PLACEBO_REPS,window=(-1,1)):
    real_pos=ret.index.searchsorted(pd.Timestamp(real_day0))
    min_pos=ESTIMATION_LENGTH+ESTIMATION_GAP+abs(window[0])+2
    max_pos=real_pos-window[1]-2
    candidates=np.arange(min_pos,max_pos)
    if not len(candidates): return np.array([])
    chosen=rng.choice(candidates,size=min(reps,len(candidates)),replace=False)
    out=[]
    for p in chosen:
        try:
            ar,_,_,_=event_ar(ret,TARGET,ret.index[p],spec,abs(window[0]),window[1])
            w=ar.loc[(ar.index>=window[0])&(ar.index<=window[1])]
            if len(w)==window[1]-window[0]+1: out.append(w.sum())
        except Exception:
            pass
    return np.asarray(out)

if not active_events.empty:
    for _,ev in active_events.iterrows():
        placebo=placebo_distribution(returns,ev["mapped_day0"])
        if len(placebo):
            fig=go.Figure(go.Histogram(x=placebo*100,nbinsx=45,name="Placebo CAR"))
            fig.add_vline(x=0,line_color="white")
            fig.update_layout(template=TEMPLATE,title=f"{ev['event_id']} | placebo [-1,+1] CARs",
                              xaxis_title="Pseudo-event CAR (%)",yaxis_title="Frequency",height=430)
            fig.show()
        else:
            print("Insufficient pre-event observations for placebo:",ev["event_id"])

## 5. Volatility, downside risk, beta and volume surprise

In [ ]:
r=returns[TARGET].dropna()
vol20=r.rolling(20).std()*np.sqrt(252)
vol60=r.rolling(60).std()*np.sqrt(252)
downside20=r.clip(upper=0).rolling(20).std()*np.sqrt(252)
beta60=returns[TARGET].rolling(60).cov(returns[MARKET])/returns[MARKET].rolling(60).var()

fig=make_subplots(rows=3,cols=1,shared_xaxes=True,vertical_spacing=.07,
                  subplot_titles=("Annualized realized volatility","60-session market beta","Downside semivolatility"))
fig.add_trace(go.Scatter(x=vol20.index,y=vol20*100,name="20-session vol",line=dict(color=C["nvda"])),row=1,col=1)
fig.add_trace(go.Scatter(x=vol60.index,y=vol60*100,name="60-session vol",line=dict(color=C["market"])),row=1,col=1)
fig.add_trace(go.Scatter(x=beta60.index,y=beta60,name="Rolling beta",line=dict(color=C["accent"])),row=2,col=1)
fig.add_trace(go.Scatter(x=downside20.index,y=downside20*100,name="Downside vol",line=dict(color=C["negative"])),row=3,col=1)
for _,ev in active_events.iterrows():
    for row in [1,2,3]:
        fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"],row=row,col=1)
fig.update_layout(template=TEMPLATE,height=820,title="NVIDIA risk-regime dashboard",hovermode="x unified")
fig.update_yaxes(title="Volatility (%)",row=1,col=1)
fig.update_yaxes(title="Beta",row=2,col=1)
fig.update_yaxes(title="Downside vol (%)",row=3,col=1)
fig.show()

In [ ]:
# Volume surprise versus trailing 20-session baseline; shift(1) prevents look-ahead.
v=volume[TARGET].astype(float)
v_mu=v.rolling(20).mean().shift(1)
v_sd=v.rolling(20).std().shift(1)
volume_z=(v-v_mu)/v_sd
dollar_turnover=volume[TARGET]*prices[TARGET]

fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.08,
                  subplot_titles=("Volume surprise Z-score","Dollar turnover"))
fig.add_trace(go.Bar(x=volume_z.index,y=volume_z,name="Volume Z",
    marker_color=np.where(volume_z.fillna(0)>=0,C["positive"],C["negative"])),row=1,col=1)
fig.add_hline(y=2,line_dash="dash",line_color=C["negative"],row=1,col=1)
fig.add_trace(go.Scatter(x=dollar_turnover.index,y=dollar_turnover/1e9,
    name="Dollar turnover ($bn)",line=dict(color=C["market"],width=2)),row=2,col=1)
for _,ev in active_events.iterrows():
    for row in [1,2]:
        fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"],row=row,col=1)
fig.update_layout(template=TEMPLATE,height=630,title="Trading activity and liquidity proxies",hovermode="x unified")
fig.update_yaxes(title="Z-score",row=1,col=1)
fig.update_yaxes(title="$ billions",row=2,col=1)
fig.show()

In [ ]:
# Full-sample GARCH(1,1): do not fit separate post-event GARCH models on a tiny post-event sample.
try:
    from arch import arch_model
    gdata=r.dropna()*100
    gfit=arch_model(gdata,mean="Constant",vol="GARCH",p=1,q=1,dist="t").fit(disp="off")
    gvol=gfit.conditional_volatility*np.sqrt(252)
    persistence=gfit.params.get("alpha[1]",np.nan)+gfit.params.get("beta[1]",np.nan)
    print(gfit.summary())
    print(f"alpha + beta persistence = {persistence:.3f}")
    fig=go.Figure(go.Scatter(x=gvol.index,y=gvol,name="Annualized conditional volatility",
                             line=dict(color=C["accent"],width=2)))
    for _,ev in active_events.iterrows():
        fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"])
    fig.update_layout(template=TEMPLATE,title="GARCH(1,1) conditional volatility",
                      yaxis_title="Annualized volatility (%)",height=460)
    fig.show()
except ImportError:
    print("Optional package missing. Install with: %pip install arch")

## 6. Cross-company risk analytics

In [ ]:
peer_cols=[t for t in STOCKS if t in returns.columns]
peer_returns=returns[peer_cols].dropna(how="all")
corr=peer_returns.corr()

fig=px.imshow(corr,text_auto=".2f",zmin=-1,zmax=1,color_continuous_scale="RdBu_r",
              template=TEMPLATE,title="Cross-company daily-return correlation")
fig.update_layout(height=640)
fig.show()

ann_ret=peer_returns.mean()*252
ann_vol=peer_returns.std()*np.sqrt(252)
risk_stats=pd.DataFrame({
    "annualized_return":ann_ret,"annualized_volatility":ann_vol,
    "return_vol_ratio":ann_ret/ann_vol,
    "worst_day":peer_returns.min(),"daily_5pct_quantile":peer_returns.quantile(.05),
    "max_drawdown":drawdowns[peer_cols].min()
}).sort_values("annualized_volatility",ascending=False)
display(risk_stats.style.format({
    "annualized_return":"{:.1%}","annualized_volatility":"{:.1%}",
    "return_vol_ratio":"{:.2f}","worst_day":"{:.1%}",
    "daily_5pct_quantile":"{:.1%}","max_drawdown":"{:.1%}"
}))

In [ ]:
fig=go.Figure()
for t in peer_cols:
    fig.add_trace(go.Scatter(x=[ann_vol[t]*100],y=[ann_ret[t]*100],mode="markers+text",
        text=[t],textposition="top center",name=t,
        marker=dict(size=14,color=C["nvda"] if t==TARGET else None)))
fig.update_layout(template=TEMPLATE,title="Annualized risk-return map",
                  xaxis_title="Annualized volatility (%)",yaxis_title="Annualized arithmetic return (%)",
                  showlegend=False,height=520)
fig.show()

fig=go.Figure()
for t in [x for x in peer_cols if x!=TARGET]:
    rc=returns[TARGET].rolling(60).corr(returns[t])
    fig.add_trace(go.Scatter(x=rc.index,y=rc,name=t))
for _,ev in active_events.iterrows():
    fig.add_vline(x=ev["mapped_day0"],line_dash="dash",line_color=C["negative"])
fig.update_layout(template=TEMPLATE,title="60-session rolling correlation with NVIDIA",
                  yaxis_title="Correlation",yaxis=dict(range=[-1,1]),height=500,hovermode="x unified")
fig.show()

## 7. CEO governance / key-person exposure index (optional, evidence-based)

The earlier notebook used arbitrary company scores. This version intentionally refuses to invent them.

Create `data/governance_inputs.csv` with columns:

`ticker, ceo_key_person_dependence, succession_readiness_gap, board_independence_gap, disclosure_uncertainty, regulatory_exposure, source_url, assessment_date`

Score each input 0–100 using a documented rubric; higher means greater exposure. Cite a source and assessment date for each company. Do not infer misconduct from appearance, rumors or private-life speculation. The composite measures governance/key-person exposure, **not** the probability of misconduct.

In [ ]:
GOV_FILE=DATA_DIR/"governance_inputs.csv"
WEIGHTS=pd.Series({
    "ceo_key_person_dependence":.25,
    "succession_readiness_gap":.25,
    "board_independence_gap":.20,
    "disclosure_uncertainty":.15,
    "regulatory_exposure":.15
})
assert np.isclose(WEIGHTS.sum(),1)

if GOV_FILE.exists():
    gov=pd.read_csv(GOV_FILE)
    required=["ticker",*WEIGHTS.index,"source_url","assessment_date"]
    missing=[x for x in required if x not in gov.columns]
    if missing: raise ValueError(f"Missing governance columns: {missing}")
    if gov[list(WEIGHTS.index)].isna().any().any(): raise ValueError("Missing scores: do not silently impute.")
    if not gov[list(WEIGHTS.index)].apply(lambda x:x.between(0,100).all()).all():
        raise ValueError("All governance scores must be 0-100.")
    if not gov["source_url"].fillna("").str.startswith(("https://","http://")).all():
        raise ValueError("Every row requires a source URL.")
    gov["CRGRI"]=gov[list(WEIGHTS.index)].dot(WEIGHTS)
    display(gov.sort_values("CRGRI",ascending=False))
else:
    print("No sourced governance_inputs.csv. Index calculation skipped rather than using arbitrary placeholder scores.")

## 8. Portfolio scenario lab: explicit assumptions, no false precision

This section is separate from the event study. It asks what a hypothetical portfolio would lose **if** specified shocks occurred. Scenario probabilities and shock sizes are assumptions unless calibrated to a defensible historical event dataset. A CRGRI score is not used as an event probability.

In [ ]:
# Example hypothetical portfolio; replace with your own clearly labeled weights.
portfolio=pd.Series({"NVDA":.30,"META":.20,"AMZN":.20,"TSLA":.15,"MSFT":.15})
portfolio=portfolio[portfolio.index.isin(returns.columns)]
if not np.isclose(portfolio.sum(),1):
    raise ValueError("Available portfolio weights must sum to 1.0.")

PORTFOLIO_VALUE=100_000
SHOCKS=np.array([.01,.03,.05,.10])
stress=pd.DataFrame({
    "shock":SHOCKS,
    "loss_usd":[PORTFOLIO_VALUE*(portfolio*s).sum() for s in SHOCKS]
})
stress["loss_pct"]=stress["loss_usd"]/PORTFOLIO_VALUE
display(stress.style.format({"shock":"{:.0%}","loss_usd":"${:,.0f}","loss_pct":"{:.1%}"}))

fig=px.bar(stress,x=stress["shock"].map(lambda x:f"{x:.0%}"),y="loss_usd",
           text_auto=",.0f",template=TEMPLATE,title="Parallel portfolio downside stress")
fig.update_layout(xaxis_title="Assumed shock applied to each holding",yaxis_title="Portfolio loss ($)")
fig.show()

In [ ]:
# Correlated Monte Carlo scenario engine. Every parameter below is an assumption.
N_SIM=30000
ANNUAL_EVENT_PROB=.02       # assumed per-name event probability
SHOCK_MEAN=-.06             # assumed conditional adverse return
SHOCK_SD=.03                # assumed dispersion of conditional shock

assets=portfolio.index.tolist()
corr_hist=returns[assets].dropna().corr().to_numpy()
eigval,eigvec=np.linalg.eigh(corr_hist)
chol=eigvec@np.diag(np.sqrt(np.clip(eigval,1e-8,None)))
z=rng.standard_normal((N_SIM,len(assets)))@chol.T
hits=rng.random((N_SIM,len(assets)))<ANNUAL_EVENT_PROB
shock=np.clip(SHOCK_MEAN+SHOCK_SD*z,-.50,0)
loss_pct=(hits*(-shock)*portfolio.to_numpy()).sum(axis=1)

var95=np.quantile(loss_pct,.95)
es95=loss_pct[loss_pct>=var95].mean()
print(f"Mean scenario loss: {loss_pct.mean():.3%}")
print(f"95% VaR: {var95:.3%} | 95% expected shortfall: {es95:.3%}")
print("These are assumption-driven scenario outputs, not forecasts or calibrated risk estimates.")

fig=go.Figure(go.Histogram(x=loss_pct*100,nbinsx=65,marker_color=C["market"],name="Scenario loss"))
fig.add_vline(x=var95*100,line_dash="dash",line_color=C["negative"],annotation_text=f"95% VaR {var95:.2%}")
fig.add_vline(x=es95*100,line_dash="dot",line_color=C["accent"],annotation_text=f"95% ES {es95:.2%}")
fig.update_layout(template=TEMPLATE,title="Simulated annual portfolio scenario losses",
                  xaxis_title="Portfolio loss (%)",yaxis_title="Frequency",height=480)
fig.show()

## 9. Optional news sentiment: FinBERT with publication-time alignment

Prepare `data/headlines.csv` with `published_at,headline,source,url`. Use original publication timestamps, not download timestamps. Map weekend/after-close headlines to the next market session. FinBERT classifies tone; it does not establish whether a claim is true or whether investors believed it. Store metadata and derived scores, not copyrighted article text.

In [ ]:
HEADLINES=DATA_DIR/"headlines.csv"
if HEADLINES.exists():
    try:
        from transformers import pipeline
        h=pd.read_csv(HEADLINES,parse_dates=["published_at"])
        required={"published_at","headline","source","url"}
        if not required.issubset(h.columns): raise ValueError(f"Required columns: {required}")
        h=h.dropna(subset=["published_at","headline"]).copy()
        classifier=pipeline("text-classification",model="ProsusAI/finbert",truncation=True)
        pred=classifier(h["headline"].astype(str).tolist(),batch_size=16)
        h["label"]=[p["label"].lower() for p in pred]
        h["confidence"]=[p["score"] for p in pred]
        h["sentiment"]=[p["score"] if p["label"].lower()=="positive"
                        else -p["score"] if p["label"].lower()=="negative" else 0 for p in pred]
        h["session"]=h["published_at"].dt.tz_localize(None).dt.normalize().map(
            lambda d:map_to_session(returns.index,d))
        daily=h.groupby("session")["sentiment"].agg(["mean","count","std"])
        display(h[["published_at","headline","source","label","confidence","sentiment","session"]].head(12))
        display(daily.tail(12))
    except ImportError:
        print("Optional dependencies: %pip install transformers torch")
else:
    print("headlines.csv not supplied; sentiment module skipped.")

## 10. Results dashboard and interpretation checklist

In [ ]:
print("DATA SNAPSHOT")
print(f"Window: {prices[TARGET].first_valid_index().date()} to {prices[TARGET].last_valid_index().date()}")
print(f"NVIDIA sample total return: {prices[TARGET].iloc[-1]/prices[TARGET].iloc[0]-1:.2%}")
print(f"NVIDIA annualized realized volatility: {returns[TARGET].std()*np.sqrt(252):.2%}")
print(f"NVIDIA maximum drawdown: {drawdowns[TARGET].min():.2%}")

if active_events.empty:
    print("\\nEVENT STUDY: no verified event rows enabled; no event-specific conclusions generated.")
elif isinstance(event_results,pd.DataFrame) and not event_results.empty:
    display(event_results[["event_id","model","window","days_used","complete_window","CAR","p_value","p_adj_BH","bootstrap_p"]])

### Interpretation checklist

Before publishing, verify:

1. Is each event date supported by an original source and mapped to the correct trading session?
2. Are all reported event windows complete? Are truncated windows clearly labeled?
3. Does NVIDIA underperform both the market and a peer-control basket?
4. Are conclusions robust across event windows and benchmark specifications?
5. Are multiple tests accounted for?
6. Did realized volatility, downside volatility and volume surprise change relative to a pre-event baseline?
7. Could AI-capex expectations, semiconductor news, export controls, earnings or leadership succession explain the move?
8. Have you avoided treating market-cap change as causal loss?

**Careful conclusion template:** “NVIDIA's observed return over the selected window was X. The market-model abnormal-return estimate was Y, while the peer-control specification estimated Z. Estimates are sensitive to benchmark choice and overlapping information; they should not be interpreted as a causal estimate of scandal-attributable value destruction.”

## 11. Export research artifacts

In [ ]:
prices.to_csv(DATA_DIR/"adjusted_prices.csv")
returns.to_csv(DATA_DIR/"daily_returns.csv")
volume.to_csv(DATA_DIR/"daily_volume.csv")
events.to_csv(DATA_DIR/"event_ledger.csv",index=False)
if isinstance(event_results,pd.DataFrame) and not event_results.empty:
    event_results.to_csv(DATA_DIR/"event_study_results.csv",index=False)
risk_stats.to_csv(DATA_DIR/"cross_company_risk_statistics.csv")
print("Saved price, return, volume, event-ledger and cross-company outputs to ./data/")
print("For static Plotly image export, install kaleido; for interactive GitHub artifacts use fig.write_html('visualizations/chart.html').")

## 12. Limitations and research extensions

**Identification:** one-firm case studies cannot isolate causal effects from concurrent news. Expected-return models create counterfactual estimates, not proof of causation. SOXX contains NVIDIA. Peer firms differ materially in business mix and governance.

**Statistical:** short post-event samples make volatility comparisons unstable. Event-induced variance and fat tails can weaken conventional event-study tests. Bootstrap results depend on block length and the historical residual distribution. Multiple windows/specifications create multiple-testing concerns.

**Data:** Yahoo Finance is convenient but not an institutional feed; adjusted prices can be revised. Archive retrieval dates and data snapshots. News timestamps and event definitions need independent verification. Governance scores need a transparent rubric and source trail.

**Next-level extensions:**
1. Intraday event study with exact announcement timestamps.
2. Historical panel of CEO departures, misconduct allegations and governance events.
3. Cross-sectional regression of abnormal returns on key-person dependence, board independence and succession readiness.
4. Matched-control difference-in-differences with pre-trend diagnostics.
5. Options-implied volatility, skew and volatility risk premium around announcements.
6. Liquidity analysis using bid-ask spreads, Amihud illiquidity and order-book depth.
7. NLP analysis using article-level text where legally available and correctly timestamped.

**Reproducibility:** record retrieval date, package versions, event-source URLs, event definitions, model parameters, random seed and manual governance-score decisions. This is independent research, not investment advice.